# TSP (Travelling Salesman Problem) — Comparison of Three ILP Models

We solve the same 25-city TSP instance using three different Integer Linear Programming formulations and compare their performance.

| Model | Subtour Elimination | Key Idea |
|-------|--------------------|--------------------|
| DFJ   | Lazy Constraints (Callback) | Dynamically add cutting planes whenever a subtour is detected |
| MTZ   | Positional variables $u_i$ | Enforce visit order via auxiliary continuous variables |
| QAP   | City-position assignment matrix | Model as a linearized quadratic assignment problem |

## Common Data

Coordinates for 25 cities are given. The edge cost between cities $i$ and $j$ is the Euclidean distance truncated to one decimal place:

$$c_{ij} = \lfloor \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2} \times 10 \rfloor / 10$$

In [ ]:
import math
import time
import gurobipy as gp
from gurobipy import GRB

coords = {
    1:  (150, 200), 2:  (41,  49),  3:  (35,  17),  4:  (55,  45),
    5:  (55,  20),  6:  (15,  30),  7:  (25,  30),  8:  (20,  50),
    9:  (10,  43),  10: (55,  60),  11: (30,  60),  12: (20,  65),
    13: (50,  35),  14: (30,  25),  15: (15,  10),  16: (30,  5),
    17: (10,  20),  18: (5,   30),  19: (18,  50),  20: (19,  48),
    21: (40,  50),  22: (8,   60),  23: (13, 143),  24: (150, 60),
    25: (90,  80),
}
nodes = list(coords.keys())
n     = len(nodes)

def euclidean(i, j):
    dx = coords[i][0] - coords[j][0]
    dy = coords[i][1] - coords[j][1]
    return math.floor(math.sqrt(dx**2 + dy**2) * 10) / 10

c = {(i, j): euclidean(i, j) for i in nodes for j in nodes if i != j}

results = {}  # stores results from each model
print(f"Cities: {n},  Edges: {len(c)}")

---
## Model 1: DFJ (Dantzig-Fulkerson-Johnson)

### Formulation

**Variables**
$$x_{ij} \in \{0, 1\} \quad \forall i \neq j$$

**Objective**
$$\min \sum_{i \neq j} c_{ij} x_{ij}$$

**Constraints**
$$\sum_{j \neq i} x_{ij} = 1 \quad \forall i \qquad \text{(exactly one arc leaves each city)}$$
$$\sum_{i \neq j} x_{ij} = 1 \quad \forall j \qquad \text{(exactly one arc enters each city)}$$
$$\sum_{i \in S} \sum_{\substack{j \in S \\ j \neq i}} x_{ij} \leq |S| - 1 \quad \forall S \subsetneq V,\; |S| \geq 2 \qquad \text{(subtour elimination)}$$

### Characteristics

- The subtour elimination constraints are **exponential in number** ($2^n$ subsets), so they cannot all be added upfront.
- Instead, **Branch-and-Cut with a Lazy Constraint Callback** is used:  
  whenever an integer solution is found, the callback detects subtours and dynamically adds the violated cuts.
- In practice, only a small fraction of the cuts are ever needed, making DFJ the **most scalable** formulation for large instances.
- Requires Gurobi's `LazyConstraints` parameter and the `cbLazy()` API.

In [ ]:
def find_subtours(x_val):
    edges   = [(i, j) for (i, j), v in x_val.items() if v > 0.5]
    nxt     = {i: j for i, j in edges}
    visited = set()
    subtours = []
    for start in nodes:
        if start in visited:
            continue
        tour, cur = [], start
        while cur not in visited:
            visited.add(cur)
            tour.append(cur)
            cur = nxt.get(cur)
            if cur is None:
                break
        if tour:
            subtours.append(tour)
    return subtours

def subtour_callback(model, where):
    if where == GRB.Callback.MIPSOL:
        x_val    = model.cbGetSolution(model._x)
        subtours = find_subtours(x_val)
        if len(subtours) > 1:
            for S in subtours:
                if len(S) < n:
                    model.cbLazy(
                        gp.quicksum(model._x[i, j] for i in S for j in S if i != j)
                        <= len(S) - 1
                    )

m1 = gp.Model("TSP_DFJ")
m1.Params.LazyConstraints = 1
m1.Params.OutputFlag      = 0

x1    = m1.addVars(c.keys(), vtype=GRB.BINARY, name="x")
m1._x = x1
m1.setObjective(gp.quicksum(c[i,j] * x1[i,j] for i,j in c), GRB.MINIMIZE)
m1.addConstrs(gp.quicksum(x1[i,j] for j in nodes if j != i) == 1 for i in nodes)
m1.addConstrs(gp.quicksum(x1[i,j] for i in nodes if i != j) == 1 for j in nodes)

t0 = time.time()
m1.optimize(subtour_callback)
elapsed1 = time.time() - t0

x1_val   = {(i,j): x1[i,j].X for i,j in c}
tour1    = find_subtours(x1_val)[0]
idx      = tour1.index(1)
tour1    = tour1[idx:] + tour1[:idx]

results['DFJ'] = {
    'obj':      m1.ObjVal,
    'time':     elapsed1,
    'vars':     m1.NumVars,
    'constrs':  m1.NumConstrs,
    'nodes':    int(m1.NodeCount),
    'tour':     tour1,
}
print(f"[DFJ]  Optimal Cost: {m1.ObjVal:.1f}")
print(f"Tour : {' -> '.join(map(str, tour1))} -> {tour1[0]}")
print(f"Time : {elapsed1:.2f}s  |  B&B Nodes: {int(m1.NodeCount)}")

---
## Model 2: MTZ (Miller-Tucker-Zemlin)

### Formulation

**Variables**
$$x_{ij} \in \{0, 1\}, \quad u_i \in [0,\, n-1] \quad \text{(continuous)}$$

**Objective and assignment constraints** — same as DFJ

**MTZ subtour elimination constraints**
$$u_i - u_j + n \cdot x_{ij} \leq n - 1 \quad \forall i \neq j,\; i,j \neq \text{depot}$$
$$u_{\text{depot}} = 0, \quad 1 \leq u_i \leq n-1 \quad \forall i \neq \text{depot}$$

### Characteristics

- $u_i$ is an auxiliary variable representing the **visit order (position)** of node $i$ in the tour.
- Subtour elimination uses only $O(n^2)$ constraints, so **all constraints can be added upfront** — no callback needed.
- However, the Big-M coefficient ($n$) weakens the **LP relaxation bound**, which can lead to a larger Branch-and-Bound search tree compared to DFJ.
- Simple to implement: works out-of-the-box with any standard MIP solver without callbacks.

In [ ]:
depot = 1
non_depot = [v for v in nodes if v != depot]

m2 = gp.Model("TSP_MTZ")
m2.Params.OutputFlag = 0

x2 = m2.addVars(c.keys(), vtype=GRB.BINARY, name="x")
u  = m2.addVars(nodes, lb=0, ub=n-1, vtype=GRB.CONTINUOUS, name="u")

m2.setObjective(gp.quicksum(c[i,j] * x2[i,j] for i,j in c), GRB.MINIMIZE)
m2.addConstrs(gp.quicksum(x2[i,j] for j in nodes if j != i) == 1 for i in nodes)
m2.addConstrs(gp.quicksum(x2[i,j] for i in nodes if i != j) == 1 for j in nodes)

m2.addConstrs(
    u[i] - u[j] + n * x2[i,j] <= n - 1
    for i in non_depot for j in non_depot if i != j
)
m2.addConstr(u[depot] == 0)
m2.addConstrs(u[i] >= 1 for i in non_depot)
m2.addConstrs(u[i] <= n - 1 for i in non_depot)

t0 = time.time()
m2.optimize()
elapsed2 = time.time() - t0

nxt   = {i: j for (i,j) in c if x2[i,j].X > 0.5}
tour2 = [depot]
cur   = nxt[depot]
while cur != depot:
    tour2.append(cur)
    cur = nxt[cur]

results['MTZ'] = {
    'obj':     m2.ObjVal,
    'time':    elapsed2,
    'vars':    m2.NumVars,
    'constrs': m2.NumConstrs,
    'nodes':   int(m2.NodeCount),
    'tour':    tour2,
}
print(f"[MTZ]  Optimal Cost: {m2.ObjVal:.1f}")
print(f"Tour : {' -> '.join(map(str, tour2))} -> {tour2[0]}")
print(f"Time : {elapsed2:.2f}s  |  B&B Nodes: {int(m2.NodeCount)}")

---
## Model 3: QAP (Quadratic Assignment Problem formulation)

### Formulation

**Variables**
$$x_{ij} \in \{0,1\} : \text{city } i \text{ is assigned to position } j$$
$$y_{ijkl} \in \{0,1\} : \text{linearization of } x_{ij} \cdot x_{kl}$$

**Objective** (distance counted only between consecutive position pairs $(j,\, j{+}1)$)
$$\min \sum_{i \neq k} \sum_{(j,l) \in \text{consec}} d_{ik} \cdot y_{ijkl}$$

**Assignment constraints**
$$\sum_j x_{ij} = 1 \; \forall i \qquad \sum_i x_{ij} = 1 \; \forall j$$

**Bilinear linearization** ($y_{ijkl} = x_{ij} \cdot x_{kl}$)
$$y_{ijkl} \geq x_{ij} + x_{kl} - 1, \quad y_{ijkl} \leq x_{ij}, \quad y_{ijkl} \leq x_{kl}$$

### Characteristics

- Subtour elimination is **implicit in the model structure**: positions are fixed as $1 \to 2 \to \cdots \to n \to 1$, so subtours are structurally impossible.
- Linearizing the bilinear term $x_{ij} \cdot x_{kl}$ requires **a large number of $y$ variables**:  
  for $n=25$, $x$: 625 variables, $y$: $\approx 25 \times 24 \times 25 = 15{,}000$ variables.
- The explosion in model size makes this formulation **impractical for large instances**.
- City 1 is fixed to position 1 to break rotational symmetry and speed up solving.

In [ ]:
positions = list(range(1, n + 1))
d         = {(i, k): euclidean(i, k) for i in nodes for k in nodes if i != k}
consec    = [(j, j % n + 1) for j in positions]

m3 = gp.Model("TSP_QAP")
m3.Params.OutputFlag = 0

x3 = m3.addVars(
    [(i, j) for i in nodes for j in positions],
    vtype=GRB.BINARY, name="x"
)
y = m3.addVars(
    [(i, j, k, l) for i in nodes for k in nodes if i != k
     for (j, l) in consec],
    vtype=GRB.BINARY, name="y"
)

m3.setObjective(
    gp.quicksum(
        d[i,k] * y[i,j,k,l]
        for i in nodes for k in nodes if i != k
        for (j,l) in consec
    ),
    GRB.MINIMIZE
)
m3.addConstrs(gp.quicksum(x3[i,j] for j in positions) == 1 for i in nodes)
m3.addConstrs(gp.quicksum(x3[i,j] for i in nodes)     == 1 for j in positions)

for i in nodes:
    for k in nodes:
        if i == k: continue
        for (j, l) in consec:
            m3.addConstr(y[i,j,k,l] >= x3[i,j] + x3[k,l] - 1)
            m3.addConstr(y[i,j,k,l] <= x3[i,j])
            m3.addConstr(y[i,j,k,l] <= x3[k,l])

m3.addConstr(x3[1, 1] == 1)

t0 = time.time()
m3.optimize()
elapsed3 = time.time() - t0

pos_to_city = {j: i for i in nodes for j in positions if x3[i,j].X > 0.5}
tour3 = [pos_to_city[j] for j in positions]

results['QAP'] = {
    'obj':     m3.ObjVal,
    'time':    elapsed3,
    'vars':    m3.NumVars,
    'constrs': m3.NumConstrs,
    'nodes':   int(m3.NodeCount),
    'tour':    tour3,
}
print(f"[QAP]  Optimal Cost: {m3.ObjVal:.1f}")
print(f"Tour : {' -> '.join(map(str, tour3))} -> {tour3[0]}")
print(f"Time : {elapsed3:.2f}s  |  B&B Nodes: {int(m3.NodeCount)}")

---
## Results Comparison

We compare whether all three models return the same optimal value, and how they differ in model size and solve time.

In [ ]:
print("=" * 72)
print(f"{'Model':<8} {'Opt. Cost':>10} {'Time (s)':>10} {'# Vars':>10} {'# Constrs':>10} {'B&B Nodes':>10}")
print("-" * 72)
for name, r in results.items():
    print(f"{name:<8} {r['obj']:>10.1f} {r['time']:>10.3f} {r['vars']:>10,} {r['constrs']:>10,} {r['nodes']:>10,}")
print("=" * 72)

print()
print("Tour sequences:")
for name, r in results.items():
    t = r['tour']
    print(f"  [{name}] {' -> '.join(map(str, t))} -> {t[0]}")

print()
objs = [r['obj'] for r in results.values()]
if len(set(objs)) == 1:
    print(f"All three models agree on the same optimal cost: {objs[0]:.1f}")
else:
    print("Warning: models returned different optimal values —", {k: v['obj'] for k, v in results.items()})